In [9]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, to_timestamp, round as spark_round
from pyspark.sql.types import DoubleType, TimestampType
import os

MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "mjyQgjfKGdn0urK0V1vm"
MINIO_SECRET_KEY = "At6cf5sGWjAGnHgUzV18mDxXulvVp9MZ03v2FxuK"
MINIO_BUCKET = "earthquake-data"
USE_SECURE = False

LAYER = "raw"
SOURCE = "earthquake"
START_DATE = "2025-01-01"
END_DATE = "2025-01-31"

pg_host = 'localhost'
pg_port = '5432'
pg_db = 'postgres'
table_name = 'ods.fct_earthquake'
pg_user = 'postgres_dwh'
pg_password = 'postgres_dwh'

In [2]:
jar_files = [
    "C:\\Users\\nicki\\spark_scripts\\jars\\postgresql-42.7.5.jar"]

In [3]:
spark = (
    SparkSession.builder
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.1,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.jars", ','.join(jar_files))
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}")
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") 
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000")
    .config("spark.hadoop.fs.s3a.connection.request.timeout", "60000") 
    .config("spark.hadoop.fs.s3a.multipart.size", "104857600") 
    .getOrCreate()
)

In [4]:
s3_path = f"s3a://{MINIO_BUCKET}/{LAYER}/{SOURCE}/{START_DATE}/{START_DATE}_00-00-00.gz.parquet"

In [5]:
df = spark.read.parquet(s3_path)

In [6]:
df.printSchema()

root
 |-- time: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- depth: string (nullable = true)
 |-- mag: string (nullable = true)
 |-- magType: string (nullable = true)
 |-- nst: string (nullable = true)
 |-- gap: string (nullable = true)
 |-- dmin: string (nullable = true)
 |-- rms: string (nullable = true)
 |-- net: string (nullable = true)
 |-- id: string (nullable = true)
 |-- updated: string (nullable = true)
 |-- place: string (nullable = true)
 |-- type: string (nullable = true)
 |-- horizontalError: string (nullable = true)
 |-- depthError: string (nullable = true)
 |-- magError: string (nullable = true)
 |-- magNst: string (nullable = true)
 |-- status: string (nullable = true)
 |-- locationSource: string (nullable = true)
 |-- magSource: string (nullable = true)



In [10]:
count_df = (
    df
    .groupBy(F.date_format(F.col("time"),"yyyy-MM-dd").alias("date"))
    .agg(
        F.count(F.col("time")).alias("cnt_d")
    )
)

In [11]:
count_df.show(truncate=False)

+----------+-----+
|date      |cnt_d|
+----------+-----+
|2025-01-01|385  |
|2025-01-02|51   |
+----------+-----+



In [20]:
spark.stop()